# ⚙️ Notebook 1: Keep Config Out of the Binary

**Goal:** understand *why* we externalise configuration and see a clear
**bad → good → best** progression in Python.

- **Hardcoded** config means every change requires a new build & deploy.
- **Externalised** config means the *same binary* runs everywhere
  (dev / stage / prod) — the environment decides behaviour.

> 💡 **12-factor rule of thumb**: *if a value changes between deployments, it's config*.


## 🛠️ Setup

```bash
cd 05-microservices/configuration-externalization
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟥 BAD: hardcoded values

Everything is baked into the source. To run this in staging you'd have to
*edit the code and redeploy*. Scary.


In [ ]:
def bad_app():
    DB_HOST = "prod-db.internal"   # 😱 how do you run this in staging?
    RATE_LIMIT = 100
    DEBUG = False
    return f"connecting to {DB_HOST}, limit={RATE_LIMIT}, debug={DEBUG}"

print(bad_app())


## 🟨 GOOD: read from environment variables

Environment variables are the simplest form of externalised config and
work everywhere (containers, CI, bare metal). We provide sensible defaults
for local development.


In [ ]:
import os

def good_app():
    db = os.environ.get("DB_HOST", "localhost")
    rl = int(os.environ.get("RATE_LIMIT", "10"))
    debug = os.environ.get("DEBUG", "false").lower() == "true"
    return f"connecting to {db}, limit={rl}, debug={debug}"

# Simulate prod vs staging *without rebuilding*.
os.environ["DB_HOST"] = "prod-db"; os.environ["RATE_LIMIT"] = "1000"; os.environ["DEBUG"] = "false"
print("prod :", good_app())

os.environ["DB_HOST"] = "stage-db"; os.environ["RATE_LIMIT"] = "50"; os.environ["DEBUG"] = "true"
print("stage:", good_app())


### ⚠️ Problems with raw `os.environ`

1. **No validation** — `RATE_LIMIT=banana` crashes at first use, not at startup.
2. **No types** — everything is a string; you manually `int(...)` / parse bools.
3. **No documentation** — which variables does the app read? Nobody knows.
4. **No precedence** — what if you want *file < env < CLI* ordering?


## 🟩 BEST: a typed, validated `Settings` object (pydantic)

We declare all config in *one place*, with types and defaults.
Pydantic validates on startup and gives a clear error if anything is wrong —
the service **fails fast** instead of crashing in production hours later.


In [ ]:
from pydantic import BaseModel, Field, ValidationError
import os

class Settings(BaseModel):
    db_host: str = Field(default="localhost", description="Database hostname")
    rate_limit: int = Field(default=10, ge=1, le=10_000, description="Requests/sec per user")
    debug: bool = Field(default=False)

    @classmethod
    def from_env(cls, env: dict[str, str] | None = None) -> "Settings":
        env = env if env is not None else os.environ
        # Simple precedence: defaults < env. A real app might also layer a config file.
        return cls(
            db_host=env.get("DB_HOST", cls.model_fields["db_host"].default),
            rate_limit=int(env.get("RATE_LIMIT", cls.model_fields["rate_limit"].default)),
            debug=env.get("DEBUG", "false").lower() == "true",
        )

prod = Settings.from_env({"DB_HOST": "prod-db", "RATE_LIMIT": "1000"})
print("prod :", prod)

# Validation catches bad values *at startup*, not on the first request that
# happens to touch them. Two different kinds of "bad":

# (a) wrong type — the classic 3am pager: RATE_LIMIT=banana deployed on Friday.
try:
    Settings.from_env({"RATE_LIMIT": "banana"})
except ValueError as e:
    print("startup rejected a non-numeric value:", e)

# (b) right type, out of range — a fat-fingered extra zero.
try:
    Settings.from_env({"RATE_LIMIT": "999999"})
except ValidationError as e:
    print("\nstartup rejected an out-of-range value:\n", e)

print("\nThe point is WHEN this happens: at process start, so the deploy fails")
print("fast and the old version keeps serving. With raw os.environ the same typo")
print("crashes on the first request that reads it — possibly hours later, in")
print("production, at partial rollout, with half the fleet already broken.")


## 🗂️ Adding a file layer: precedence `defaults < file < env`

Real apps usually have a config file for the *shape* of the config and
environment variables for the few things that change per deployment
(secrets, hostnames). Environment variables **override** the file.


In [ ]:
import json, tempfile, os, pathlib

# Pretend this file ships with the app (checked into git, no secrets inside).
config_file = pathlib.Path(tempfile.gettempdir()) / "app.json"
config_file.write_text(json.dumps({"db_host": "file-db", "rate_limit": 25}))

def load_settings(path: pathlib.Path, env: dict[str, str]) -> Settings:
    data: dict = {}
    if path.exists():
        data.update(json.loads(path.read_text()))
    # env overrides file
    if "DB_HOST" in env:     data["db_host"] = env["DB_HOST"]
    if "RATE_LIMIT" in env:  data["rate_limit"] = int(env["RATE_LIMIT"])
    if "DEBUG" in env:       data["debug"] = env["DEBUG"].lower() == "true"
    return Settings(**data)

print("file only        :", load_settings(config_file, {}))
print("env overrides    :", load_settings(config_file, {"DB_HOST": "env-db"}))
print("env + file merged:", load_settings(config_file, {"RATE_LIMIT": "77"}))


## 📦 Production-ready: `pydantic-settings`

Writing the `from_env` / `load_settings` plumbing by hand is fine for teaching.
In real code most Python services use [`pydantic-settings`](https://docs.pydantic.dev/latest/concepts/pydantic_settings/) —
same `Settings` idea, but it reads env vars, `.env` files, and secrets
*automatically* with a clear precedence order.


In [ ]:
from pydantic import Field
from pydantic_settings import BaseSettings, SettingsConfigDict

class AppSettings(BaseSettings):
    # model_config tells pydantic-settings where to look and how to map names
    model_config = SettingsConfigDict(
        env_prefix="APP_",      # APP_DB_HOST -> db_host
        env_file=None,          # set to ".env" in a real project
        extra="ignore",
    )
    db_host: str = "localhost"
    rate_limit: int = Field(default=10, ge=1, le=10_000)
    debug: bool = False

# Demo: pass env in-memory via _env so we don't touch the real environment.
import os
os.environ["APP_DB_HOST"] = "prod-db"
os.environ["APP_RATE_LIMIT"] = "500"
os.environ["APP_DEBUG"] = "true"

s = AppSettings()
print("loaded:", s)
# Clean up so later cells/notebooks aren't affected.
for k in ("APP_DB_HOST", "APP_RATE_LIMIT", "APP_DEBUG"):
    os.environ.pop(k, None)


### 🧭 Precedence (what overrides what)

`pydantic-settings` layers sources with a well-defined order (highest wins):

1. **CLI / init kwargs** you pass to `AppSettings(...)`
2. **Environment variables** (optionally prefixed)
3. **`.env` file** (via `env_file=".env"`)
4. **Secrets directory** (e.g. Docker/K8s mounts in `/run/secrets/<name>`)
5. **Field defaults** in the class

That's the same *defaults < file < env* pattern we built by hand — just
battle-tested and typed.


## 🧠 Takeaways

| Step | Where config lives | Validated? | Redeploy to change? |
|------|--------------------|-----------|---------------------|
| BAD  | in code            | n/a       | yes 😬              |
| GOOD | env vars           | no        | restart only        |
| BEST | env vars + file, typed `Settings` (or `BaseSettings`) | yes ✅ | restart only |

Next notebook: **feature flags**, which let us change behaviour
*without even restarting*.
